# Agente RAG - Challenge ALURA ORACLE OCI
Fluxo:

**5 PDFs → carregamento → chunking → embeddings → vector store → 1 ferramenta de busca → agente → resposta**

## 1. Instalação

In [ ]:
%pip install -qU pypdf langchain langchain-community langchain-huggingface langchain-groq langchain-text-splitters sentence-transformers

## 2. Configuração

As chaves ficam em variáveis de ambiente.

In [ ]:
import os

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError(
        "Defina a variável de ambiente GROQ_API_KEY antes de executar o notebook."
    )

## 3. Modelo de linguagem

O LLM será usado somente para interpretar a pergunta, decidir quando usar a ferramenta e formular a resposta final.

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

## 4. Carregando os 5 PDFs

Todos os documentos ficam em `documentos/` e são colocados na **mesma coleção**.
O retriever encontrará os trechos semanticamente mais próximos da pergunta.

In [ ]:
from pathlib import Path

from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

DOCUMENTOS_DIR = Path("documentos")

pdfs = sorted(DOCUMENTOS_DIR.glob("*.pdf"))

if len(pdfs) != 5:
    raise ValueError(
        f"Esperados 5 PDFs em '{DOCUMENTOS_DIR}', mas foram encontrados {len(pdfs)}."
    )

for pdf in pdfs:
    print(pdf.name)

loader = DirectoryLoader(
    str(DOCUMENTOS_DIR),
    glob="*.pdf",
    loader_cls=PyPDFLoader,
    show_progress=True
)

pages = loader.load()

print(f"Total de páginas/documentos carregados: {len(pages)}")

## 5. Dividindo os documentos em chunks

Chunks menores ajudam a devolver somente o trecho relevante ao LLM, reduzindo contexto desnecessário e, consequentemente, tokens.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = splitter.split_documents(pages)

print(f"Total de chunks: {len(chunks)}")

## 6. Embeddings

Usamos um modelo multilíngue, adequado aos documentos e perguntas em português.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embed_model = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-small"
)

## 7. Criando uma única base vetorial

A base contém os chunks dos cinco PDFs.

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore.from_documents(
    chunks,
    embedding=embed_model
)

retriever = vector_store.as_retriever(
    search_kwargs={"k": 2}
)

print("Vector store criada com sucesso.")

## 8. Testando a recuperação

Antes de criar o agente, verificamos se a busca encontra trechos relevantes.

In [ ]:
query = "Como fazer commits pequenos e descritivos?"

docs = retriever.invoke(query)

for i, doc in enumerate(docs, start=1):
    print(f"--- Resultado {i} ---")
    print(f"Arquivo: {doc.metadata.get('source')}")
    print(f"Página: {doc.metadata.get('page')}")
    print(doc.page_content[:800])
    print()